<a href="https://colab.research.google.com/github/saurabhsen848/Deep-Learning/blob/main/Project_6_Deep_LearningForward_Propagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 6: Simple Neural Network Forward Propagation

**Objective:** Use the Iris dataset to understand how a neural network processes input data and produces predictions through forward propagation.


## Task 1: Load and prepare the Iris dataset

The Iris dataset has 4 input features (sepal length, sepal width, petal length, petal width) and 3 flower species classes (setosa, versicolor, virginica).


In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Load dataset
iris = load_iris()
X = iris.data          # shape: (150, 4)
y = iris.target        # shape: (150,)  -> values 0, 1, 2

print("Feature names:", iris.feature_names)
print("Target classes:", iris.target_names)
print("X shape:", X.shape, "| y shape:", y.shape)


Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target classes: ['setosa' 'versicolor' 'virginica']
X shape: (150, 4) | y shape: (150,)


In [2]:
# Standardize features (helps the network process values on a similar scale)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Train samples:", X_train.shape[0], "| Test samples:", X_test.shape[0])


Train samples: 120 | Test samples: 30


## Task 2: Build a simple neural network and perform forward propagation

We'll build a small network **from scratch using NumPy**, so the forward propagation steps are fully visible:

- Input layer: 4 features
- Hidden layer: 5 neurons
- Output layer: 3 neurons (one per Iris class)

Weights are randomly initialized (this network is not trained — the assignment only asks us to observe forward propagation, not train the model).


In [3]:
np.random.seed(42)

n_input = 4      # 4 input features
n_hidden = 5      # 5 hidden neurons
n_output = 3      # 3 output classes

# Randomly initialize weights and biases
W1 = np.random.randn(n_input, n_hidden) * 0.5
b1 = np.zeros((1, n_hidden))

W2 = np.random.randn(n_hidden, n_output) * 0.5
b2 = np.zeros((1, n_output))

print("W1 shape:", W1.shape, "| b1 shape:", b1.shape)
print("W2 shape:", W2.shape, "| b2 shape:", b2.shape)


W1 shape: (4, 5) | b1 shape: (1, 5)
W2 shape: (5, 3) | b2 shape: (1, 3)


In [4]:
def forward_propagation(X, W1, b1, W2, b2):
    # Layer 1: input -> hidden
    z1 = np.dot(X, W1) + b1
    a1 = relu(z1)

    # Layer 2: hidden -> output
    z2 = np.dot(a1, W2) + b2
    a2 = softmax(z2)

    return a2


## Task 3: Apply activation functions and display the predicted output

We use **ReLU** in the hidden layer and **Softmax** in the output layer, since this is a multi-class classification problem.


In [5]:
def relu(z):
    return np.maximum(0, z)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))  # for numerical stability
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


In [6]:
# Run forward propagation on the test set
predictions_prob = forward_propagation(X_test, W1, b1, W2, b2)

# Predicted class = index with the highest probability
predicted_classes = np.argmax(predictions_prob, axis=1)

print("Predicted probabilities for first 5 test samples:")
print(np.round(predictions_prob[:5], 3))

print("\nPredicted classes for first 5 test samples:", predicted_classes[:5])
print("Actual classes for first 5 test samples:      ", y_test[:5])


Predicted probabilities for first 5 test samples:
[[0.337 0.331 0.332]
 [0.023 0.014 0.962]
 [0.248 0.446 0.306]
 [0.329 0.338 0.332]
 [0.293 0.3   0.407]]

Predicted classes for first 5 test samples: [0 2 1 1 2]
Actual classes for first 5 test samples:       [1 0 2 1 1]


In [7]:
import pandas as pd

results_df = pd.DataFrame({
    "Actual Species": [iris.target_names[i] for i in y_test],
    "Predicted Species": [iris.target_names[i] for i in predicted_classes],
    "Prob(setosa)": np.round(predictions_prob[:, 0], 3),
    "Prob(versicolor)": np.round(predictions_prob[:, 1], 3),
    "Prob(virginica)": np.round(predictions_prob[:, 2], 3),
})

results_df.head(10)


,Actual Species,Predicted Species,Prob(setosa),Prob(versicolor),Prob(virginica)
0,versicolor,setosa,0.337,0.331,0.332
1,setosa,virginica,0.023,0.014,0.962
2,virginica,versicolor,0.248,0.446,0.306
3,versicolor,versicolor,0.329,0.338,0.332
4,versicolor,virginica,0.293,0.300,0.407
5,setosa,virginica,0.048,0.019,0.933
6,versicolor,setosa,0.333,0.333,0.333
7,virginica,versicolor,0.248,0.446,0.306
8,versicolor,virginica,0.332,0.288,0.380
9,versicolor,virginica,0.336,0.315,0.349


**Note:** Since the network's weights are randomly initialized and not trained, predicted classes will likely not match the actual species accurately. That's expected — the goal of this assignment is to understand the *mechanics* of forward propagation (how inputs flow through weights, biases, and activation functions to produce an output), not to build an accurate classifier.

If you also want to see a *trained* version for comparison, run the optional cell below.


## Optional: Train the network for a more meaningful prediction (bonus, not required)

In [8]:
# Simple training loop using gradient descent (optional extension)
def one_hot(y, n_classes):
    m = y.shape[0]
    one_hot_y = np.zeros((m, n_classes))
    one_hot_y[np.arange(m), y] = 1
    return one_hot_y

y_train_oh = one_hot(y_train, 3)
lr = 0.1

for epoch in range(1000):
    # Forward pass
    z1 = np.dot(X_train, W1) + b1
    a1 = relu(z1)
    z2 = np.dot(a1, W2) + b2
    a2 = softmax(z2)

    # Backward pass (simple gradient descent)
    m = X_train.shape[0]
    dz2 = a2 - y_train_oh
    dW2 = np.dot(a1.T, dz2) / m
    db2 = np.sum(dz2, axis=0, keepdims=True) / m

    dz1 = np.dot(dz2, W2.T) * (z1 > 0)
    dW1 = np.dot(X_train.T, dz1) / m
    db1 = np.sum(dz1, axis=0, keepdims=True) / m

    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

# Evaluate after training
trained_preds = forward_propagation(X_test, W1, b1, W2, b2)
trained_classes = np.argmax(trained_preds, axis=1)
accuracy = np.mean(trained_classes == y_test)
print(f"Accuracy after training: {accuracy:.2%}")


Accuracy after training: 100.00%


## Summary

- **Task 1:** Loaded and standardized the Iris dataset, split into train/test sets.
- **Task 2:** Built a small neural network (4 → 5 → 3) from scratch with NumPy and implemented forward propagation.
- **Task 3:** Applied ReLU (hidden layer) and Softmax (output layer) activation functions, and displayed predicted class probabilities and predicted species for test samples.

This demonstrates the core forward propagation process: input data is multiplied by weights, shifted by biases, passed through an activation function at each layer, and the final layer's output is converted into class probabilities.
